# 05｜四模型 34 個 X 席位與 X/Y 研究資料建立

**執行環境：Google Colab（Python 3）**。會讀取 `janusChatGPT` 研究資料及使用 GitHub `main` 的**固定 SHA**，結果只寫入研究 run／報告目錄。不會修改來源 canonical、Risk ACTIVE、GCP、Registry、PG 或 Core/Mart。

請由上往下執行每一個 cell；首次使用先完成 Google Drive OAuth。


In [ ]:
# 此設定每支 Notebook 可獨立執行；中途更換 Colab Runtime 也能接續同一研究 run。
import sys, os, json, hashlib, subprocess, traceback
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
assert ROOT.is_dir(), '沒有找到給ai專用/janusChatGPT；不能改用個人用資料夾'
DATASET = ROOT/'資料源'/'JanusDataset_v3_202610'
MARKET = ROOT/'資料源'/'大盤產業_免費來源_2025H1_試跑'
RUN_HOME = ROOT/'模型研究'/'_colab_pipeline_runs'
ACTIVE = RUN_HOME/'ACTIVE_RUN.json'
REPO = Path('/content/janus-omniforge')
REPO_URL = 'https://github.com/tommylin15/janus-omniforge.git'
RUN_HOME.mkdir(parents=True, exist_ok=True)

def sh(*args, check=True, capture=False):
    return subprocess.run([str(x) for x in args],check=check,text=True,
                          capture_output=capture)

def sha_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return h.hexdigest()

def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str,allow_nan=False)+'\n',encoding='utf-8')
    tmp.replace(path)

def read_json(path):return json.loads(Path(path).read_text(encoding='utf-8'))

def packages():
    # Colab 預裝 pandas/numpy/scikit-learn；只有缺少依賴才安裝。不引入付費服務。
    from importlib.util import find_spec
    missing=[(module,pip) for module,pip in [('pyarrow','pyarrow'),('sklearn','scikit-learn'),('xgboost','xgboost')]
             if find_spec(module) is None]
    if missing:sh(sys.executable,'-m','pip','install','-q',*[p for _,p in missing])
    import pandas, numpy, pyarrow, sklearn, xgboost
    return {k:__import__(k).__version__ for k in ['pandas','numpy','pyarrow','sklearn','xgboost']}

def repo_git(*args):
    # 優先使用公開存取；若 repo 是 private，使用 Colab Secrets 的 GITHUB_TOKEN。
    cmd=['git',*[str(x) for x in args]]
    env=dict(os.environ,GIT_TERMINAL_PROMPT='0')
    result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    if result.returncode!=0:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
        if not token:raise RuntimeError('GIT_ACCESS_FAILED：repo 可能是 private；請設 Colab Secret GITHUB_TOKEN，或檢查存取權')
        import tempfile
        with tempfile.TemporaryDirectory() as folder:
            ask=Path(folder)/'askpass.sh'
            ask.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token;; *) printf %s "$GITHUB_TOKEN";; esac\n')
            ask.chmod(0o700)
            env.update(GIT_ASKPASS=str(ask),GITHUB_TOKEN=token)
            result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    if result.returncode:raise RuntimeError('GIT_COMMAND_FAILED：'+result.stderr[-500:])
    return result

def checkout(code_sha=None):
    if not (REPO/'.git').is_dir():repo_git('clone',REPO_URL,REPO)
    else:repo_git('-C',REPO,'fetch','origin','main')
    if code_sha:
        # 不能用最新 main 偷換固定研究程式版本。
        existing=sh('git','-C',REPO,'cat-file','-e',code_sha+'^{commit}',check=False)
        if existing.returncode:repo_git('-C',REPO,'fetch','origin',code_sha)
        repo_git('-C',REPO,'checkout','--detach','--force',code_sha)
    else:repo_git('-C',REPO,'checkout','--detach','--force','origin/main')
    actual=sh('git','-C',REPO,'rev-parse','HEAD',capture=True).stdout.strip()
    if code_sha and actual!=code_sha:raise RuntimeError('CODE_SHA_CHANGED')
    assert (REPO/'research/chatgpt/label_four_models.py').is_file()
    assert (REPO/'research/chatgpt/janus_train.py').is_file()
    code=str(REPO/'research/chatgpt')
    if code not in sys.path:sys.path.insert(0,code)
    return actual

def open_run():
    if not ACTIVE.is_file():raise RuntimeError('先執行 03_收檔驗收與同步.ipynb，建立研究 run')
    cfg=read_json(ACTIVE)
    out=RUN_HOME/cfg['run_id']
    if not out.is_dir():raise RuntimeError('RUN_FOLDER_MISSING')
    if not (DATASET/'canonical/CURRENT.json').is_file():raise RuntimeError('CURRENT_MISSING')
    if sha_file(DATASET/'canonical/CURRENT.json')!=cfg['current_sha256']:
        raise RuntimeError('CURRENT_CHANGED：來源版本已切換，請勿混用研究產物')
    manifest=DATASET/cfg['manifest_rel']
    if sha_file(manifest)!=cfg['manifest_sha256']:
        raise RuntimeError('MANIFEST_CHANGED：固定 release 有變動')
    checkout(cfg['git_sha'])
    return cfg,out

def save_stage(out,n,status,details):
    obj={'stage':n,'status':status,'updated_at_utc':datetime.now(timezone.utc).isoformat(),
         'publication_allowed':False,'details':details}
    write_json(Path(out)/f'{n}_status.json',obj)
    print(json.dumps(obj,ensure_ascii=False,indent=2,default=str)[:12000]);return obj

def stage(out,n):
    p=Path(out)/f'{n}_status.json'
    return read_json(p) if p.is_file() else {'stage':n,'status':'NOT_RUN'}

def ensure_stage(out,n,allow=('PASS',)):
    status=stage(out,n)['status']
    if status not in allow:raise RuntimeError(f'PREVIOUS_GATE_{n}={status}; allowed={allow}')

def keep_current(cfg):
    if sha_file(DATASET/'canonical/CURRENT.json')!=cfg['current_sha256']:
        raise RuntimeError('CURRENT_CHANGED_DURING_STAGE')

def fail_stage(out,n,exc):
    save_stage(out,n,'BLOCKED',{'error_type':type(exc).__name__, 'message':str(exc)[:1200]})
    raise exc

versions=packages()
print('Dependency versions:',versions)


In [ ]:
# 05：運行既有 label_four_models.run (產生不可發佈研究資料)，再用同一個
# janus_precision.training_panel 定義輸出 34 個 X 席位的角色型 X/Y。
cfg,out=open_run();ensure_stage(out,'04',('PASS','PARTIAL'))
assert stage(out,'04')['details']['dataset_mechanical']=='PASS', 'LOADER_NOT_VERIFIED'
import pandas as pd
import numpy as np
import janus_precision as precision
from label_four_models import run as label_run

try:
    result=label_run(DATASET,out/'labels',verify_only=False)
    write_json(out/'05_label_result.json',result)
    assert result['publication_allowed'] is False
    roles={'quant':['ohlcv'],'risk':['ohlcv'],'fundamental':['revenue'],
           'valuation':['valuation','financials','ohlcv']}
    role_features={'risk':precision.RISK,'quant':precision.QUANT,
                   'fundamental':precision.FUND,'valuation':precision.VAL}
    assert sum(map(len,role_features.values()))==34
    # 估值財務 mapping 並非憑空猜測；此處沒有經認證設定就不建立假的 VAL X。
    financial_mapping_file=ROOT/'模型研究'/'_config'/'valuation_financial_mapping.json'
    financial_mapping=(read_json(financial_mapping_file) if financial_mapping_file.is_file() else None)
    slot_report={}
    for role,kinds in roles.items():
        role_folder=out/'xy'/role;role_folder.mkdir(parents=True,exist_ok=True)
        if role=='valuation' and not (financial_mapping and financial_mapping.get('metrics') and financial_mapping.get('cohorts')):
            slot_report[role]={'status':'BLOCKED_VERIFIED_MAPPING_REQUIRED','slots':len(role_features[role]),
                'mapping_location':str(financial_mapping_file),'computed_X_coverage':'NOT_VERIFIED'}
            continue
        data,receipt=precision.load_release(DATASET,kinds=kinds)
        panel,features=precision.training_panel(data,role,'retrospective',20,financial_mapping)
        keys=precision.KEY+['signal_group']
        assert list(features)==list(role_features[role])
        # 與現有 janus_train.py 所輸出的 X/Y 欄位、排序完全一致。
        x=panel[keys+features]
        ycols=[c for c in ['y','y_pe','y_pb'] if c in panel]
        y=panel[keys+['signal_at','mature_at','reason']+ycols]
        assert not set(x).intersection(ycols+['mature_at','reason','signal_at'])
        x.to_parquet(role_folder/'X.parquet',index=False)
        y.to_parquet(role_folder/'Y_NOT_X.parquet',index=False)
        obs=pd.to_numeric(panel['y'],errors='coerce') if 'y' in panel else pd.Series(np.nan,index=panel.index)
        valid=panel.reason.eq('OK') & x[features].notna().all(axis=1)
        report={'status':'RESEARCH_DERIVED','role':role,'rows':len(panel),
                'features':features,'x_slots':len(features),
                'non_null_x_rows':int(x[features].notna().all(axis=1).sum()),
                'non_null_x_rate':float(x[features].notna().all(axis=1).mean()),
                'eligible_x_reason_ok':int(valid.sum()),
                'mature_at_known_rows':int(panel.mature_at.notna().sum()),
                'y_known_rows':int(obs.notna().sum()),
                'reason_counts':{str(k):int(v) for k,v in panel.reason.value_counts().items()},
                'historical_pit':'NOT_VERIFIED','publication_allowed':False,
                'X_sha256':sha_file(role_folder/'X.parquet'),
                'Y_sha256':sha_file(role_folder/'Y_NOT_X.parquet')}
        write_json(role_folder/'receipt.json',report)
        slot_report[role]=report
        del data,panel,x,y
    keep_current(cfg)
    write_json(out/'05_feature_slots.json',slot_report)
    stage_status='PASS' if all(v.get('status')=='RESEARCH_DERIVED' for v in slot_report.values()) else 'PARTIAL'
    save_stage(out,'05',stage_status,{'roles':{k:v['status'] for k,v in slot_report.items()},
                                      'total_slots_defined':34,'historical_pit':'NOT_VERIFIED'})
except Exception as exc:fail_stage(out,'05',exc)
